⏱️ **Time required:** ~3 minutes | **Type:** Domain pipeline (run all cells)

# 📣 RideFlow Marketing — Domain Pipeline

**Domain Owner:** Growth Team  
**Systems:** google_analytics · hubspot · meta_ads · google_ads  

This notebook represents the **Marketing domain team's** autonomous pipeline. Like Operations, it demonstrates a **multi-system domain** — the Growth team owns data from 4 different MarTech vendors. Each system produces campaign, attribution, and engagement data that feeds into a unified marketing analytics layer.

| System | Bronze Entities | Silver Entities | Gold Entities |
| :-- | :-- | :-- | :-- |
| **Google Analytics** | `sessions`, `app_events` | `silver_google_analytics_sessions` | — |
| **HubSpot** | `email_campaigns`, `push_notifications` | `silver_hubspot_email_campaigns` | — |
| **Meta Ads** | `campaigns` | `silver_meta_ads_campaigns` | — |
| **Google Ads** | `campaigns` | `silver_google_ads_campaigns` | `gold_google_ads_acquisition_roi` |

> **Cross-Domain Dependency:** This pipeline reads `gold_rideflow_rider_lifetime_value` from the Marketplace domain to correlate ad spend with actual rider value. Run `07b_rideflow_marketplace.ipynb` first.

---
## Step 1 · Environment Setup

In [ ]:
import os
import sys
from pathlib import Path
import polars as pl
import lakelogic as ll

PROJECT_ROOT = Path(".").resolve()
LAKEHOUSE = PROJECT_ROOT / "lakehouse"
ENV = "local"

print(f"Project Root : {PROJECT_ROOT}")
print(f"Lakehouse    : {LAKEHOUSE}")

### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

---
## Step 2 · Load Marketing Registries (Multi-System)

The Marketing domain owns 4 independent systems. Each gets its own `DomainRegistry`, but they share domain-level config from `_domain.yaml` (SLOs, compliance erasure strategy: `redact`, observatory).

In [ ]:
from lakelogic.core.registry import DomainRegistry

mkt_base = PROJECT_ROOT / "assets" / "domains_rideflow" / "marketing"

registries = {}
for system_name in ["google_analytics", "hubspot", "meta_ads", "google_ads"]:
    sys_yaml = mkt_base / system_name / "_system.yaml"
    registries[system_name] = DomainRegistry.from_yaml(str(sys_yaml))
    contracts = registries[system_name].get_active_contracts()
    print(f"\n📦 {system_name} — {len(contracts)} active contracts")
    for c in contracts:
        print(f"  [{c.layer:6s}] {c.entity}")

---
## Step 3 · Generate Synthetic Marketing Data

We generate realistic data for all 4 systems, referencing rider IDs from the Marketplace domain for attribution analysis.

In [ ]:
import random
import string
import json
from datetime import datetime, timezone, timedelta

# Read Marketplace rider IDs for attribution cross-references
marketplace_riders_path = LAKEHOUSE / "marketplace" / "silver" / "silver_rideflow_rider_profiles"

if marketplace_riders_path.exists():
    riders_df = pl.read_delta(str(marketplace_riders_path))
    rider_ids = riders_df.select("rider_id").unique().to_series().to_list()
    print(f"✅ Loaded {len(rider_ids)} rider IDs from Marketplace domain")
else:
    print("⚠️ Marketplace Silver riders not found. Using synthetic placeholders.")
    rider_ids = [f"RDR-{i:06d}" for i in range(300)]

In [ ]:
def gen_id(prefix, length=8):
    return f"{prefix}_{''.join(random.choices(string.ascii_lowercase + string.digits, k=length))}"


def gen_email(name):
    domain = random.choice(["gmail.com", "yahoo.com", "outlook.com", "icloud.com"])
    return f"{name.lower().replace(' ', '.')}{random.randint(1, 99)}@{domain}"


now = datetime.now(timezone.utc)
first_names = ["James", "Maria", "David", "Sarah", "Michael", "Jessica", "Carlos", "Emily", "Ahmed", "Priya"]
last_names = ["Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis", "Rodriguez", "Martinez"]
campaign_names = [
    "Summer Rides Promo",
    "New User Welcome",
    "Airport Pickup Launch",
    "Weekend Warriors",
    "Corporate Commute",
    "Holiday Season 2025",
    "Rider Reactivation Q1",
    "Driver Recruitment UK",
    "Refer a Friend",
    "Safety First Campaign",
]

# ── Google Analytics Sessions ────────────────────────────────────────────
n_sessions = 2000
ga_sessions = []
for i in range(n_sessions):
    session_start = now - timedelta(hours=random.randint(1, 720))
    user_id = random.choice(rider_ids[:100]) if random.random() > 0.4 else ""  # 60% auth

    ga_sessions.append(
        {
            "session_id": gen_id("GA", 12),
            "user_pseudo_id": gen_id("CID", 16),
            "user_id": user_id,
            "ip_address": f"{random.randint(1, 255)}.{random.randint(0, 255)}.{random.randint(0, 255)}.{random.randint(1, 254)}",
            "session_start": session_start.isoformat(),
            "session_duration_seconds": str(random.randint(5, 1800)),
            "page_views": str(random.randint(1, 25)),
            "traffic_source": random.choices(
                ["google", "direct", "facebook", "instagram", "email", "referral"], weights=[35, 25, 15, 10, 10, 5]
            )[0],
            "traffic_medium": random.choices(
                ["organic", "cpc", "email", "social", "(none)"], weights=[30, 25, 15, 20, 10]
            )[0],
            "traffic_campaign": random.choice(campaign_names) if random.random() > 0.4 else "",
            "device_category": random.choices(["mobile", "desktop", "tablet"], weights=[65, 28, 7])[0],
            "geo_country": random.choices(
                ["United Kingdom", "United States", "Germany", "France", "India"], weights=[55, 20, 10, 8, 7]
            )[0],
            "landing_page": random.choice(
                ["/", "/download", "/pricing", "/drivers", "/about", "/promo/summer", "/safety"]
            ),
        }
    )

# ── Google Analytics App Events ──────────────────────────────────────────
n_app_events = 3000
ga_app_events = []
event_names = [
    "app_open",
    "ride_requested",
    "ride_completed",
    "payment_confirmed",
    "surge_viewed",
    "promo_applied",
    "driver_rated",
    "ride_cancelled",
]
for i in range(n_app_events):
    ts = now - timedelta(hours=random.randint(1, 720))
    ga_app_events.append(
        {
            "event_id": gen_id("EVT", 12),
            "user_pseudo_id": gen_id("CID", 16),
            "user_id": random.choice(rider_ids[:100]) if random.random() > 0.3 else "",
            "event_name": random.choices(event_names, weights=[20, 18, 15, 12, 10, 10, 10, 5])[0],
            "event_timestamp": ts.isoformat(),
            "platform": random.choices(["ios", "android", "web"], weights=[52, 40, 8])[0],
            "app_version": random.choice(["4.12.0", "4.11.3", "4.11.2", "4.10.1"]),
            "geo_country": random.choices(["UK", "US", "DE", "FR", "IN"], weights=[55, 20, 10, 8, 7])[0],
        }
    )

# ── HubSpot Email Campaigns ──────────────────────────────────────────────
n_emails = 1500
hs_emails = []
email_subjects = [
    "🚗 Your ride credits are waiting!",
    "Ride FREE this weekend with code SUMMER25",
    "We miss you — here's 20% off your next ride",
    "New: Airport pickup now live in your city!",
    "Safety update: See what's new in RideFlow",
    "You've earned Gold status! 🌟",
]
for i in range(n_emails):
    fname = random.choice(first_names)
    lname = random.choice(last_names)
    sent = now - timedelta(hours=random.randint(1, 720))
    opened = (sent + timedelta(hours=random.randint(1, 48))) if random.random() > 0.35 else None
    clicked = (opened + timedelta(minutes=random.randint(1, 60))) if opened and random.random() > 0.6 else None

    hs_emails.append(
        {
            "campaign_id": gen_id("HS"),
            "recipient_email": gen_email(f"{fname} {lname}"),
            "recipient_name": f"{fname} {lname}",
            "campaign_name": random.choice(campaign_names),
            "subject_line": random.choice(email_subjects),
            "sent_at": sent.isoformat(),
            "opened_at": opened.isoformat() if opened else "",
            "clicked_at": clicked.isoformat() if clicked else "",
            "bounced": str(random.random() < 0.02).lower(),
            "unsubscribed": str(random.random() < 0.01).lower(),
        }
    )

# ── HubSpot Push Notifications ───────────────────────────────────────────
n_pushes = 800
hs_pushes = []
push_titles = [
    "Your ride is arriving!",
    "Rate your driver",
    "Weekend promo: 25% off",
    "New feature: Schedule rides",
    "Surge pricing alert",
]
for i in range(n_pushes):
    sent = now - timedelta(hours=random.randint(1, 720))
    hs_pushes.append(
        {
            "notification_id": gen_id("PUSH"),
            "user_id": random.choice(rider_ids[:100]),
            "title": random.choice(push_titles),
            "sent_at": sent.isoformat(),
            "opened": str(random.random() > 0.4).lower(),
            "platform": random.choices(["ios", "android"], weights=[55, 45])[0],
        }
    )

# ── Meta Ads Campaigns ───────────────────────────────────────────────────
n_meta = 200
meta_ads = []
for i in range(n_meta):
    d = now - timedelta(days=random.randint(0, 90))
    impressions = random.randint(1000, 50000)
    clicks = int(impressions * random.uniform(0.005, 0.04))
    conversions = int(clicks * random.uniform(0.02, 0.15))

    meta_ads.append(
        {
            "campaign_id": gen_id("META"),
            "campaign_name": random.choice(campaign_names),
            "adset_id": gen_id("AS"),
            "adset_name": random.choice(["UK_18-34_Mobile", "US_25-44_All", "UK_35-54_Desktop", "EU_18-24_iOS"]),
            "date": d.strftime("%Y-%m-%d"),
            "impressions": str(impressions),
            "clicks": str(clicks),
            "spend": f"{random.uniform(50, 2000):.2f}",
            "actions_json": json.dumps([{"action_type": "app_install", "value": str(conversions)}]),
            "conversions": str(conversions),
            "currency": random.choice(["GBP", "USD", "EUR"]),
        }
    )

# ── Google Ads Campaigns ─────────────────────────────────────────────────
n_gads = 200
g_ads = []
for i in range(n_gads):
    d = now - timedelta(days=random.randint(0, 90))
    impressions = random.randint(500, 30000)
    clicks = int(impressions * random.uniform(0.01, 0.06))
    conversions = int(clicks * random.uniform(0.03, 0.12))
    cost = clicks * random.uniform(0.5, 3.0)
    conv_value = conversions * random.uniform(8, 45)

    g_ads.append(
        {
            "campaign_id": gen_id("GADS"),
            "campaign_name": random.choice(campaign_names),
            "ad_group_id": gen_id("AG"),
            "ad_group_name": random.choice(["Brand_Exact", "Competitor_Broad", "Generic_Rideshare", "Airport_Rides"]),
            "date": d.strftime("%Y-%m-%d"),
            "impressions": str(impressions),
            "clicks": str(clicks),
            "cost": str(round(cost * 100)),
            "conversions": str(conversions),
            "conversion_value": f"{conv_value:.2f}",
            "currency": random.choice(["GBP", "USD"]),
        }
    )

df_ga_sessions = pl.DataFrame(ga_sessions)
df_ga_app_events = pl.DataFrame(ga_app_events)
df_hs_emails = pl.DataFrame(hs_emails)
df_hs_pushes = pl.DataFrame(hs_pushes)
df_meta_ads = pl.DataFrame(meta_ads)
df_g_ads = pl.DataFrame(g_ads)

print("Generated:")
print(f"  🔍 {len(ga_sessions):,} GA sessions")
print(f"  📱 {len(ga_app_events):,} GA app events")
print(f"  📧 {len(hs_emails):,} HubSpot email campaigns")
print(f"  🔔 {len(hs_pushes):,} HubSpot push notifications")
print(f"  📘 {len(meta_ads):,} Meta ad campaign rows")
print(f"  🔎 {len(g_ads):,} Google Ads campaign rows")

---
## Step 4 · Land Synthetic Data

Write each system's data into its own landing zone.

In [ ]:
landing_root = LAKEHOUSE / "_data" / "marketing"

datasets = [
    ("google_analytics", "sessions", df_ga_sessions),
    ("google_analytics", "app_events", df_ga_app_events),
    ("hubspot", "email_campaigns", df_hs_emails),
    ("hubspot", "push_notifications", df_hs_pushes),
    ("meta_ads", "campaigns", df_meta_ads),
    ("google_ads", "campaigns", df_g_ads),
]

for system, entity, df in datasets:
    dest = landing_root / system / entity
    dest.mkdir(parents=True, exist_ok=True)
    csv_path = dest / f"{entity}.csv"
    df.write_csv(str(csv_path))
    print(f"  ✅ {system}/{entity}: {len(df):,} rows → {csv_path.relative_to(LAKEHOUSE)}")

---
## Step 5 · Run Pipeline Per System

In [ ]:
from lakelogic.pipeline.runner import LakehousePipeline

summaries = {}
for system_name, registry in registries.items():
    print(f"\n{'=' * 60}")
    print(f"📣 Running pipeline: marketing / {system_name}")
    print(f"{'=' * 60}")

    runner = LakehousePipeline(registry, engine=ENGINE)

    summary = runner.run(target_layers="bronze,silver,gold", dry_run=False, environment=ENV)
    summaries[system_name] = summary
    print(summary)

---
## Step 6 · Channel Mix Attribution

Combine ad spend from Meta and Google Ads to produce a unified channel mix view — a key marketing analytics use case.

In [ ]:
# Read Silver ad data from both platforms
meta_path = LAKEHOUSE / "marketing" / "silver" / "silver_meta_ads_campaigns"
gads_path = LAKEHOUSE / "marketing" / "silver" / "silver_google_ads_campaigns"

meta_silver = pl.read_delta(str(meta_path)) if meta_path.exists() else df_meta_ads
gads_silver = pl.read_delta(str(gads_path)) if gads_path.exists() else df_g_ads

# Normalize to comparable schema
meta_spend = meta_silver.select(
    [
        pl.lit("Meta").alias("channel"),
        pl.col("spend").cast(pl.Float64).alias("spend"),
        pl.col("impressions").cast(pl.Int64).alias("impressions"),
        pl.col("clicks").cast(pl.Int64).alias("clicks"),
        pl.col("conversions").cast(pl.Int64).alias("conversions"),
    ]
)

gads_spend = gads_silver.select(
    [
        pl.lit("Google Ads").alias("channel"),
        (pl.col("cost").cast(pl.Float64) / 100).alias("spend"),
        pl.col("impressions").cast(pl.Int64).alias("impressions"),
        pl.col("clicks").cast(pl.Int64).alias("clicks"),
        pl.col("conversions").cast(pl.Int64).alias("conversions"),
    ]
)

channel_mix = (
    pl.concat([meta_spend, gads_spend])
    .group_by("channel")
    .agg(
        [
            pl.col("spend").sum().alias("total_spend"),
            pl.col("impressions").sum().alias("total_impressions"),
            pl.col("clicks").sum().alias("total_clicks"),
            pl.col("conversions").sum().alias("total_conversions"),
        ]
    )
    .with_columns(
        [
            (pl.col("total_spend") / pl.col("total_conversions")).round(2).alias("cost_per_conversion"),
            (pl.col("total_clicks") / pl.col("total_impressions") * 100).round(2).alias("ctr_pct"),
        ]
    )
)

print("📊 Channel Mix Attribution")
display(channel_mix)

---
## Step 7 · Email Campaign Performance

Analyze HubSpot email campaign engagement rates.

In [ ]:
email_path = LAKEHOUSE / "marketing" / "silver" / "silver_hubspot_email_campaigns"

if email_path.exists():
    emails = pl.read_delta(str(email_path))
else:
    emails = df_hs_emails
    print("⚠️ Using in-memory email data")

campaign_perf = (
    emails.group_by("campaign_name")
    .agg(
        [
            pl.count().alias("sends"),
            pl.col("opened_at").filter(pl.col("opened_at") != "").count().alias("opens"),
            pl.col("clicked_at").filter(pl.col("clicked_at") != "").count().alias("clicks"),
            pl.col("bounced").filter(pl.col("bounced") == "true").count().alias("bounces"),
            pl.col("unsubscribed").filter(pl.col("unsubscribed") == "true").count().alias("unsubs"),
        ]
    )
    .with_columns(
        [
            (pl.col("opens") / pl.col("sends") * 100).round(1).alias("open_rate_pct"),
            (pl.col("clicks") / pl.col("sends") * 100).round(1).alias("click_rate_pct"),
        ]
    )
    .sort("sends", descending=True)
)

print("📧 Email Campaign Performance")
display(campaign_perf)

---
## ✅ Data Products Published

The Marketing domain pipeline has completed across all 4 systems. Published data products:

| Data Product | System | Consumers |
| :-- | :-- | :-- |
| `silver_google_analytics_sessions` | GA4 | Web analytics, funnel analysis |
| `silver_hubspot_email_campaigns` | HubSpot | CRM reporting, lifecycle marketing |
| `silver_meta_ads_campaigns` | Meta | Paid social ROI, attribution |
| `silver_google_ads_campaigns` | Google Ads | Paid search ROI, keyword analytics |
| `gold_google_ads_acquisition_roi` | Google Ads | Cross-domain rider acquisition cost |

### Multi-System Architecture

```
Marketing Domain (Growth Team)
┌─────────────────────────────────────────────────────────────┐
│                                                             │
│  ┌─────────┐  ┌─────────┐  ┌──────────┐  ┌────────────┐  │
│  │   GA4   │  │ HubSpot │  │ Meta Ads │  │ Google Ads │  │
│  │(session)│  │ (email) │  │  (ads)   │  │   (ads)    │  │
│  └────┬────┘  └────┬────┘  └────┬─────┘  └─────┬──────┘  │
│       │             │            │              │          │
│  B → S          B → S        B → S         B → S → G     │
│                                                             │
│  Shared: _domain.yaml (SLO, compliance [redact], cost)     │
└─────────────────────────────────────────────────────────────┘
        │
        ▼
  Marketplace
  rider_lifetime_value
  (data product)
```

### Next Notebooks

- **`07g_compliance_gdpr_rtbf.ipynb`** — Cross-domain privacy erasure
- **`07i_data_mesh_dashboards.ipynb`** — Unified mesh observability